# 🍊 Trabajo Final IPDI: Detección y Estimación de Producción de Naranjas
## Sistema de Entrenamiento Distribuido (GitHub + Drive)

**Flujo de Trabajo:**
1. **Código:** Se descarga fresco desde GitHub (rama `rama_leom` o la que uses).
2. **Datos:** Se inyectan desde Google Drive (`dataset.zip`) al disco local de Colab.
3. **Ejecución:** Se entrena en GPU T4 y los resultados se guardan en Drive.

---

### 1. Configuración de Credenciales y Rutas
Define aquí dónde están tus cosas.

In [ ]:
# --- USUARIO: CONFIGURA ESTO UNA VEZ ---

# 1. URL de tu repositorio GitHub
REPO_URL = "https://github.com/LeonEspinosa/IPDI_TrabajoFinal_G10_YOLO.git"
BRANCH = "rama_leom"  # La rama donde estás trabajando

# 2. Ruta de tu dataset en Google Drive
# Ubicación: Mi unidad > IPDI > Trabajo Final > dataset.zip
DRIVE_DATASET_PATH = "/content/drive/MyDrive/IPDI/Trabajo Final/dataset.zip"

# 3. Carpeta para guardar los resultados del entrenamiento
# Los guardaremos en una subcarpeta 'Resultados_YOLO' dentro de tu carpeta de trabajo
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/IPDI/Trabajo Final/Resultados_YOLO"

print("✅ Configuración cargada. Apuntando a 'Trabajo Final'.")

### 2. Montaje y Clonación
Conectamos los discos duros y traemos el cerebro (código).

In [ ]:
from google.colab import drive
import os
import shutil

# 1. Montar Drive
drive.mount('/content/drive')

# 2. Clonar/Actualizar Repositorio
REPO_NAME = REPO_URL.split("/")[-1].replace(".git", "")
CODE_DIR = f"/content/{REPO_NAME}"

if os.path.exists(CODE_DIR):
    print("🔄 Repositorio detectado. Actualizando código (git pull)...")
    %cd {CODE_DIR}
    !git checkout {BRANCH}
    !git pull
else:
    print("⬇️ Clonando repositorio desde cero...")
    %cd /content
    !git clone -b {BRANCH} {REPO_URL}
    %cd {CODE_DIR}

print(f"✅ Código listo en: {os.getcwd()}")

# 3. Instalación de Dependencias del Repo
!pip install -r requirements.txt
!pip install ultralytics  # Asegurar que YOLO esté instalado

### 3. Ingesta de Datos (Data Pipeline)
Traemos el zip de Drive y lo descomprimimos en el entorno local efímero (/content/dataset) para máxima velocidad de lectura.

In [ ]:
import yaml

LOCAL_DATA_DIR = "/content/dataset_naranjas"

def prepare_data():
    if os.path.exists(LOCAL_DATA_DIR):
        print("ℹ️ Datos ya descomprimidos en local.")
        return
    
    if not os.path.exists(DRIVE_DATASET_PATH):
        raise FileNotFoundError(f"❌ No encuentro el dataset en: {DRIVE_DATASET_PATH}. ¡Verifica la ruta en Drive!")

    print(f"⏳ Descomprimiendo {DRIVE_DATASET_PATH}... (Esto puede tardar unos segundos)")
    shutil.unpack_archive(DRIVE_DATASET_PATH, LOCAL_DATA_DIR)
    print("✅ Datos listos.")

    # --- FIX DEL YAML ---
    # Buscamos y corregimos el data.yaml para que apunte a las rutas de Colab
    # Asumimos que dentro del zip hay un data.yaml
    potential_yamls = [f for f in os.listdir(LOCAL_DATA_DIR) if f.endswith('.yaml')]
    if not potential_yamls:
        # Si no está en la raíz, buscamos recursivamente o asumimos nombre estándar
        print("⚠️ Buscando data.yaml en subcarpetas...")
        for root, dirs, files in os.walk(LOCAL_DATA_DIR):
            for file in files:
                if file.endswith(".yaml"):
                     potential_yamls.append(os.path.join(root, file))
    
    if potential_yamls:
        target_yaml = os.path.join(LOCAL_DATA_DIR, potential_yamls[0])
        print(f"🔧 Ajustando rutas en: {target_yaml}")
        
        with open(target_yaml, 'r') as f:
            data_cfg = yaml.safe_load(f)
        
        # RUTA MÁGICA: Forzamos rutas absolutas de Colab
        data_cfg['path'] = LOCAL_DATA_DIR
        data_cfg['train'] = 'train/images'
        data_cfg['val'] = 'valid/images'
        data_cfg['test'] = 'test/images'
        
        with open(target_yaml, 'w') as f:
            yaml.dump(data_cfg, f)
        
        return target_yaml
    else:
        raise FileNotFoundError("❌ No encontré ningún archivo .yaml en el dataset descomprimido.")

# Ejecutar preparación
final_yaml_path = prepare_data()
print(f"📂 Configuración de datos lista en: {final_yaml_path}")

### 4. Ejecución del Entrenamiento
Lanzamos el script `main.py` pasando la ruta dinámica del dataset.

In [ ]:
# Crear carpeta de salida en Drive si no existe
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)

# Ejecutar Main
# --data: Le pasamos la ruta absoluta del yaml que acabamos de corregir
!python main.py --data "{final_yaml_path}"

# --- GUARDADO DE EMERGENCIA ---
# Al terminar, copiamos los resultados de Colab (volátil) a Drive (persistente)
print("💾 Respaldando resultados a Drive...")
!cp -r runs/ "{DRIVE_OUTPUT_DIR}/"
print("✅ Respaldo completado en 'IPDI/Trabajo Final/Resultados_YOLO'.")